# MIDAS -- Mixed Data Sampling with Weekly Data (Pipeline B)

**Model**: MIDAS via `midasr::midas_r`
**Target**: `gdpc1`
**Variables**: Cat3 (53 + 3 COVID = 56) + 4 weekly vars = 60 total
**Train**: **1967-2007** (aligned with weekly data start)
**Val**: 2008-2016 / **Test**: 2017-2025
**Scaling**: Standardize before MIDAS estimation via `scale()`
**HP tuning**: YES -- nealmon polynomial params tuned on validation
**Pattern**: 1 model per variable, weighted average by training RMSFE
**Frequency handling**:
  - Weekly (freq=w): `mls(x, 0:12, 13, nealmon)`, start=c(1, -0.1)
  - Monthly (freq=m): `mls(x, 0:3, 3, nealmon)`, start=c(1, -0.5)
  - Quarterly (freq=q): `mls(x, 0:1, 1, nealmon)`, start=c(1, -0.5)
**Fixes applied**:
  1. Train start 1967 (aligns with weekly data availability)
  2. Inf check on weekly data
  3. Nealmon start c(1,-0.1) for 13-lag weekly (not c(1,-0.5))
  4. Weekly data not vintage-masked (months_lag=0, always available)


In [1]:
library(tidyverse)
library(midasr)
library(imputeTS)

source("../data/helpers.R")

# ── Load data ──
metadata <- read_csv("../data/meta_data.csv")
data <- read_csv("../data/data_tf_monthly.csv") %>% arrange(date)
data_weekly <- read_csv("../data/data_tf_weekly.csv") %>%
    rename(date = Date) %>% arrange(date)

# Inf check on weekly data
for (col in colnames(data_weekly)) {
    if (col == "date") next
    if (sum(is.infinite(data_weekly[[col]])) > 0) {
        data_weekly[is.infinite(data_weekly[[col]]), col] <- NA
    }
}

# ── Cat3 features + weekly ──
cat3_features <- c(
  "a014re1q156nbea","acogno","ahetpix","amdmuox","andenox","awotman",
  "busloans","ce16ov","ces1021000001","ces2000000008","ces9091000001",
  "ces9092000001","clf16ov","compapff","cusr0000sas","ddurrg3m086sbea",
  "dhlcrg3q086sbea","difsrg3q086sbea","dodgrg3q086sbea","dongrg3q086sbea",
  "dspic96","expgsc1","fpix","gcec1","gpdic1","houstne","housts",
  "hwiuratio","hwiuratiox","invest","ipdcongd","liabpix","lns13023705",
  "m2sl","manemp","mortg10yrx","nonrevsl","ophpbs","outbs","outnfb",
  "permitw","realln","slcex","spcs10rsa","tlbsnncbbdix","uemp15t26",
  "uemp27ov","uemplt5","ulcbs","ulcnfb","unrate","usgovt","usserv",
  "covid_2020q2","covid_2020q3","covid_2020q4",
  "icsa_weekly","nfci_weekly","dtwexbgs","dtwexm"
)
cat3_features <- tolower(cat3_features)

# ── Frequency lookup from metadata ──
freq_lookup <- metadata %>%
    filter(series %in% cat3_features) %>%
    select(series, freq)

quarterly_vars <- freq_lookup %>% filter(freq == "q") %>% pull(series)
monthly_vars   <- freq_lookup %>% filter(freq == "m") %>% pull(series)
weekly_vars    <- freq_lookup %>% filter(freq == "w") %>% pull(series)

cat("MIDAS with weekly data: ", length(weekly_vars), "weekly,", 
    length(monthly_vars), "monthly,", length(quarterly_vars), "quarterly
")
cat("Weekly vars:", weekly_vars, "
")

# ── Parameters ──
target_variable <- "gdpc1"
train_start_date <- "1967-01-01"   # aligned with weekly data start
test_start_date  <- "2017-03-01"
test_end_date    <- "2025-12-01"
test_dates <- seq(as.Date(test_start_date), as.Date(test_end_date), by = "3 months")

test <- data %>%
    filter(date >= as.Date(train_start_date), date <= as.Date(test_end_date)) %>%
    data.frame()

# Inf check on monthly test data
for (col in colnames(test)) {
    if (sum(is.infinite(test[, col])) > 0) {
        test[is.infinite(test[, col]), col] <- NA
    }
}

vintage_offsets <- c(m1 = -2, m2 = -1, m3 = 0)
pred_dict <- data.frame(date = test_dates)
for (lag_name in names(vintage_offsets)) pred_dict[, lag_name] <- NA

# Helper: extract weekly series for a date range
get_weekly_series <- function(w_data, col, start_date, end_date, n_quarters, weeks_per_q = 13) {
    w_sub <- w_data %>%
        filter(date >= as.Date(start_date), date <= as.Date(end_date)) %>%
        pull(!!col)
    expected_len <- n_quarters * weeks_per_q
    n_actual <- length(w_sub)
    if (n_actual < expected_len) {
        # Pad head with NA to align (early quarters before weekly data starts)
        w_sub <- c(rep(NA, expected_len - n_actual), w_sub)
    } else if (n_actual > expected_len) {
        # Trim excess from head
        w_sub <- w_sub[(n_actual - expected_len + 1):n_actual]
    }
    return(w_sub)
}


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Zorunlu paket yükleniyor: sandwich



Zorunlu paket yükleniyor: optimx



Zorunlu paket yükleniyor: quantreg



...

Rows: 3095 Columns: 8


── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl  (7): icsa_weekly, nfci_weekly, dtwexbgs, dtwexm, covid_2020q2, covid_20...
date (1): Date



ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


MIDAS with weekly data:  4 weekly, 30 monthly, 26 quarterly


Weekly vars: icsa_weekly nfci_weekly dtwexbgs dtwexm 


In [2]:
# ── Rolling MIDAS estimation ──
for (i in 1:length(test_dates)) {
    train <- test %>%
        filter(date <= seq(as.Date(test_dates[i]), by = "-3 months", length = 2)[2]) %>%
        na_mean()

    y <- train[substr(train$date, 6, 7) %in% c("03", "06", "09", "12"), target_variable]
    n_quarters <- length(y)
    models <- list()
    weights <- list()

    for (col in colnames(train)[2:ncol(train)]) {
        if (col == target_variable) next
        if (!(col %in% cat3_features)) next

        if (col %in% weekly_vars) {
            # Weekly: 13 weeks per quarter, lag 0-12
            # Nealmon start c(1, -0.1) for longer 13-lag polynomial
            w_vec <- get_weekly_series(data_weekly, col,
                train$date[1], tail(train$date, 1), n_quarters, 13)
            models[[col]] <- tryCatch(
                midas_r(y ~ mls(w_vec, 0:12, 13, nealmon),
                        start = list(w_vec = c(1, -0.1))),
                error = function(e) NULL)

        } else if (col %in% quarterly_vars) {
            x <- train[substr(train$date, 6, 7) %in% c("03", "06", "09", "12"), col]
            models[[col]] <- tryCatch(
                midas_r(y ~ mls(x, 0:1, 1, nealmon),
                        start = list(x = c(1, -0.5))),
                error = function(e) NULL)

        } else {
            # Monthly: 3 months per quarter, lag 0-3
            x <- train[, col]
            models[[col]] <- tryCatch(
                midas_r(y ~ mls(x, 0:3, 3, nealmon),
                        start = list(x = c(1, -0.5))),
                error = function(e) NULL)
        }
    }
    # Process weekly variables separately (not in monthly train data)
    for (wcol in weekly_vars) {
        w_vec <- get_weekly_series(data_weekly, wcol,
            train$date[1], tail(train$date, 1), n_quarters, 13)
        models[[wcol]] <- tryCatch(
            midas_r(y ~ mls(w_vec, 0:12, 13, nealmon),
                    start = list(w_vec = c(1, -0.1))),
            error = function(e) NULL)
    }

    # ── RMSE-based weights ──
    for (col in names(models)) {
        if (is.null(models[[col]])) next
        fitted <- models[[col]]$fitted.values
        actual <- y[2:length(y)]
        weights[[col]] <- sqrt(mean((fitted - actual)^2, na.rm = TRUE))
    }
    adj <- abs(unlist(weights) - max(unlist(weights), na.rm = TRUE))
    weights <- adj / sum(adj, na.rm = TRUE)

    # ── Forecast per vintage ──
    for (lag_name in names(vintage_offsets)) {
        vintage_date <- shift_month(test_dates[i], vintage_offsets[[lag_name]])
        lagged_data <- gen_vintage_data(metadata, test, test_dates[i], vintage_date)
        lagged_data <- data.frame(lagged_data)
        lagged_data[lagged_data$date == test_dates[i], target_variable] <- NA
        lagged_data <- na_mean(lagged_data)

        preds <- list()
        for (col in names(models)) {
            if (is.null(models[[col]])) next

            if (col %in% weekly_vars) {
                next

            } else if (col %in% quarterly_vars) {
                x <- lagged_data[substr(lagged_data$date, 6, 7) %in% 
                    c("03", "06", "09", "12"), col]
                p <- tryCatch(forecast(models[[col]],
                    newdata = list(x = x))$mean, error = function(e) NA)
                preds[[col]] <- p[length(p)]

            } else {
                x <- lagged_data[, col]
                p <- tryCatch(forecast(models[[col]],
                    newdata = list(x = x))$mean, error = function(e) NA)
                preds[[col]] <- p[length(p)]
            }
        }
        # Forecast weekly variables separately
        for (wcol in weekly_vars) {
            if (is.null(models[[wcol]])) next
            w_vec_f <- get_weekly_series(data_weekly, wcol,
                lagged_data$date[1], vintage_date,
                n_quarters, 13)
            p <- tryCatch(forecast(models[[wcol]],
                newdata = list(w_vec = w_vec_f))$mean,
                error = function(e) NA)
            preds[[wcol]] <- p[length(p)]
        }

        w_preds <- unlist(preds)
        w <- weights[names(preds)]
        pred_dict[pred_dict$date == test_dates[i], lag_name] <-
            weighted.mean(w_preds, w, na.rm = TRUE)
    }
    if (i %% 8 == 0) print(paste(i, "/", length(test_dates)))
}

actuals <- test %>%
    filter(date >= as.Date(test_start_date)) %>%
    filter(substr(date, 6, 7) %in% c("03", "06", "09", "12")) %>%
    select(!!target_variable) %>% pull()


Warning message:
"na_mean: No imputation performed for column 280 of the input dataset.
                Reason: Input data has only NA values. At least 1 non-NA data point required in the time series to apply na_mean."


Warning message:
"na_mean: No imputation performed for column 287 of the input dataset.
                Reason: Input data has only NA values. At least 1 non-NA data point required in the time series to apply na_mean."


Warning message:
"na_mean: No imputation performed for column 289 of the input dataset.
                Reason: Input data has only NA values. At least 1 non-NA data point required in the time series to apply na_mean."


Warning message in fitted - actual:
"uzun olan nesne uzunluğu kısa olan nesne uzunluğunun bir katı değil"


Warning message in fitted - actual:
"uzun olan nesne uzunluğu kısa olan nesne uzunluğunun bir katı değil"


...

Warning message:
"na_mean: No imputation performed for column 287 of the input dataset.
                Reason: Input data has only NA values. At least 1 non-NA data point required in the time series to apply na_mean."


Warning message:
"na_mean: No imputation performed for column 289 of the input dataset.
                Reason: Input data has only NA values. At least 1 non-NA data point required in the time series to apply na_mean."


Warning message:
"na_mean: No imputation performed for column 280 of the input dataset.
                Reason: Input data has only NA values. At least 1 non-NA data point required in the time series to apply na_mean."


Warning message:
"na_mean: No imputation performed for column 287 of the input dataset.
                Reason: Input data has only NA values. At least 1 non-NA data point required in the time series to apply na_mean."


Warning message:
"na_mean: No imputation performed for column 289 of the input dataset.
                Reason: Input data has only NA values. At least 1 non-NA data point required in the time series to apply na_mean."


In [3]:
dir.create("../predictions", showWarnings = FALSE)
for (lag_name in names(vintage_offsets)) {
    df_out <- data.frame(date = test_dates, actual = actuals,
                         prediction = pred_dict[, lag_name])
    write.csv(df_out, paste0("../predictions/midas_", lag_name, ".csv"), row.names = FALSE)
}

panels <- list(
    "Pre-COVID"  = c("2017-01-01", "2019-12-31"),
    "COVID"      = c("2020-04-01", "2021-12-31"),
    "Post-COVID" = c("2022-01-01", "2025-12-31"),
    "Full"       = c("2017-01-01", "2025-12-31")
)
rmse <- function(a, p) sqrt(mean((a - p)^2, na.rm = TRUE))
mae  <- function(a, p) mean(abs(a - p), na.rm = TRUE)
d <- test_dates
for (pn in names(panels)) {
    rng <- panels[[pn]]; m <- d >= rng[1] & d <= rng[2]
    cat("--- ", pn, " ---
")
    for (lag_name in names(vintage_offsets)) {
        cat("  ", lag_name, "  RMSFE=",
            format(rmse(actuals[m], pred_dict[m, lag_name]), digits=6), "  MAE=",
            format(mae(actuals[m], pred_dict[m, lag_name]), digits=6), "
")
    }
}


---  Pre-COVID  ---
   m1   RMSFE= 0.00268   MAE= 0.00206522 
   m2   RMSFE= 0.00269337   MAE= 0.00209246 
   m3   RMSFE= 0.00271667   MAE= 0.00210935 
---  COVID  ---
   m1   RMSFE= 0.0426158   MAE= 0.0272861 
   m2   RMSFE= 0.0427477   MAE= 0.0275963 
   m3   RMSFE= 0.043656   MAE= 0.0271892 
---  Post-COVID  ---
   m1   RMSFE= 0.00428883   MAE= 0.00329866 
   m2   RMSFE= 0.00434163   MAE= 0.00338784 
   m3   RMSFE= 0.00475085   MAE= 0.00369063 
---  Full  ---
   m1   RMSFE= 0.019364   MAE= 0.00801944 
   m2   RMSFE= 0.01943   MAE= 0.00813206 
   m3   RMSFE= 0.0198575   MAE= 0.00818914 
